# 004_structured_data_complete.ipynb — Amazon Bedrock

Extrae datos de un artículo forzando una herramienta específica con toolChoice.

Ejecuta las celdas en orden con credenciales AWS y acceso al modelo. La configuración usa servicios reales. [Configuración](../README.md).

In [ ]:
# Cliente real de Amazon Bedrock: credenciales AWS del entorno o perfil.
import os
from pathlib import Path

import boto3
from dotenv import load_dotenv

repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "BEDROCK" / "README.md").is_file()
)
bedrock_root = repo_root / "BEDROCK"
load_dotenv(bedrock_root / ".env", override=False)
exercise_dir = bedrock_root / "ToolsBuildingWithApi"
os.chdir(exercise_dir)

session = boto3.Session(
    profile_name=os.getenv("AWS_PROFILE"),
    region_name=os.getenv("AWS_REGION", "us-west-2"),
)
client = session.client("bedrock-runtime")
model_id = os.getenv("BEDROCK_MODEL_ID", "us.anthropic.claude-sonnet-4-6")


In [ ]:
# Mensajes y respuestas en el formato nativo de Converse.
def message_content(content):
    if isinstance(content, str):
        if not content.strip():
            raise ValueError("El texto del mensaje no puede estar vacío.")
        return [{"text": content}]
    if not isinstance(content, list) or not content:
        raise ValueError("El mensaje necesita una lista de bloques de contenido.")
    for block in content:
        if "text" in block and not block["text"].strip():
            raise ValueError("Los bloques de texto no pueden estar vacíos.")
    return content


def add_user_message(messages, content):
    messages.append({"role": "user", "content": message_content(content)})


def add_assistant_message(messages, content):
    messages.append({"role": "assistant", "content": message_content(content)})


def chat(
    messages,
    system=None,
    stop_sequences=None,
    tools=None,
    tool_choice="auto",
    max_tokens=4000,
):
    params = {
        "modelId": model_id,
        "messages": messages,
        "inferenceConfig": {"maxTokens": max_tokens},
    }
    if stop_sequences:
        params["inferenceConfig"]["stopSequences"] = list(stop_sequences)
    if system:
        params["system"] = [{"text": system}]
    if tools:
        choices = {"auto": {"auto": {}}, "any": {"any": {}}}
        choice = choices.get(tool_choice, {"tool": {"name": tool_choice}})
        selected_tools = list(tools)
        params["toolConfig"] = {"tools": selected_tools, "toolChoice": choice}
    response = client.converse(**params)
    parts = response["output"]["message"]["content"]
    text = "\n".join(part["text"] for part in parts if "text" in part)
    return {
        "parts": parts,
        "stop_reason": response["stopReason"],
        "text": text,
        "usage": response["usage"],
    }


In [ ]:
# Tool Schemas

article_details_schema = {
    "toolSpec": {
        "name": "article_details",
        "description": "This tool should be called with details about an article. It accepts information about the article's title, author, and related topics.",
        "inputSchema": {
            "json": {
                "type": "object",
                "properties": {
                    "title": {
                        "type": "string",
                        "description": "The title of the article. Can be left empty.",
                    },
                    "author": {
                        "type": "string",
                        "description": "The name of the article's author. Can be left empty.",
                    },
                    "topics": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "A list of topics or categories that the article covers. Can be an empty list.",
                    },
                },
            }
        },
    }
}

to_json_schema = {
    "toolSpec": {
        "name": "to_json",
        "description": "This tool processes any JSON data and can be used for generating structured content, transforming information, or creating any JSON-based output needed for your task.",
        "inputSchema": {
            "json": {"type": "object", "additionalProperties": True}
        },
    }
}

In [ ]:
messages = []

add_user_message(
    messages,
    "Write a one-paragraph scholarly article about computer science. Include a title and author name",
)

result = chat(messages)

add_assistant_message(messages, result["parts"])

result["text"]


In [ ]:
messages = []

add_user_message(
    messages,
    f"""
Analyze the article below and extract key data. Then call the article_details tool.
                 
<article_text>
{result["text"]}                 
</article_text>
""",
)

json_result = chat(
    messages, tools=[article_details_schema], tool_choice="article_details"
)

json_result

In [ ]:
messages = []

add_user_message(
    messages,
    f"""
Analyze the article below and extract key data. Then call the to_json tool.

<article_text>
{result["text"]}                 
</article_text>

When you call to_json, pass in the following structure:
{{
    "title": str # title of the article,
    "author": str # author of the article,
    "topics": List[str] # List of topics mentioned in the article,
    "num_topics": int # Number of topics mentioned
}}
""",
)

flexible_result = chat(messages, tools=[to_json_schema], tool_choice="to_json")


In [ ]:
flexible_result